# Phase 2 — Iris Preprocessing Pipeline

Demonstrates the full preprocessing pipeline:

**Original** → **Denoised + Circles** → **Unrolled Strip (64×512)** → **Model Input (128×128)**

One sample is shown from each CASIA-IrisV4 subset.

In [ ]:
import sys
import os
import glob

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Ensure src/ is importable from the notebooks/ directory
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
sys.path.insert(0, os.path.abspath('..'))

from src.preprocessing.segmentation import (
    denoise_image,
    segment_iris,
    normalize_iris,
    scale_pixels,
)

In [ ]:
# ---------------------------------------------------------------------------
# Dataset subset roots (relative to project root, i.e. one level up)
# ---------------------------------------------------------------------------
PROJECT_ROOT = os.path.abspath('..')

SUBSETS = {
    "CASIA-Iris-Interval": os.path.join(PROJECT_ROOT, 'data', 'raw', 'CASIA-Iris-Interval', 'CASIA-Iris-Interval'),
    "CASIA-Iris-Lamp":     os.path.join(PROJECT_ROOT, 'data', 'raw', 'CASIA-Iris-Lamp',     'CASIA-Iris-Lamp'),
    "CASIA-Iris-Thousand": os.path.join(PROJECT_ROOT, 'data', 'raw', 'CASIA-Iris-Thousand', 'CASIA-Iris-Thousand'),
    "CASIA-Iris-Syn":      os.path.join(PROJECT_ROOT, 'data', 'raw', 'CASIA-Iris-Syn'),
}


def find_sample(subset_root: str, limit: int = 100):
    """Return the first image path from subset_root where segmentation succeeds."""
    candidates = glob.glob(os.path.join(subset_root, '**', '*.jpg'), recursive=True)[:limit]
    for path in candidates:
        denoised = denoise_image(path)
        circles  = segment_iris(denoised)
        if circles is not None:
            return path, denoised, circles
    return None, None, None


print('Subset roots:')
for name, root in SUBSETS.items():
    exists = os.path.isdir(root)
    print(f'  {name}: {"OK" if exists else "NOT FOUND"}')

In [ ]:
# ---------------------------------------------------------------------------
# Helper: run full pipeline and return all intermediate results
# ---------------------------------------------------------------------------
def run_pipeline(img_path: str, denoised: np.ndarray, circles: dict):
    original   = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    normalized = normalize_iris(denoised, circles, circles)
    tensor     = scale_pixels(normalized)
    return original, denoised, normalized, tensor


def draw_circles(image: np.ndarray, circles: dict) -> np.ndarray:
    """Draw pupil and iris circles on a copy of image (converted to BGR for colour)."""
    vis = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    cx, cy   = circles['center']
    r_pupil  = int(circles['r_pupil'])
    r_iris   = int(circles['r_iris'])
    cv2.circle(vis, (cx, cy), r_pupil, (0, 255, 0), 2)   # pupil  — green
    cv2.circle(vis, (cx, cy), r_iris,  (0, 0, 255), 2)   # iris   — red
    cv2.circle(vis, (cx, cy), 3,       (255, 0, 0), -1)  # centre — blue dot
    return vis


def plot_pipeline(subset_name: str, img_path: str, original, denoised, circles, normalized, tensor):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(f'{subset_name}\n{os.path.basename(img_path)}', fontsize=11)

    # Panel 1 — Original
    axes[0].imshow(original, cmap='gray')
    axes[0].set_title('1. Original')
    axes[0].axis('off')

    # Panel 2 — Denoised with detected circles
    circle_vis = draw_circles(denoised, circles)
    axes[1].imshow(cv2.cvtColor(circle_vis, cv2.COLOR_BGR2RGB))
    axes[1].set_title(
        f'2. Detected Circles\npupil r={int(circles["r_pupil"])}  iris r={int(circles["r_iris"])}'
    )
    axes[1].axis('off')

    # Panel 3 — Normalised rubber-sheet strip
    axes[2].imshow(normalized, cmap='gray', aspect='auto')
    axes[2].set_title(f'3. Rubber-Sheet Strip\n{normalized.shape[0]}×{normalized.shape[1]}')
    axes[2].axis('off')

    # Panel 4 — Final model input (float, [0,1])
    axes[3].imshow(tensor[:, :, 0], cmap='gray', vmin=0.0, vmax=1.0)
    axes[3].set_title(f'4. Model Input {tensor.shape}\nfloat32 [{tensor.min():.2f}, {tensor.max():.2f}]')
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Run pipeline on one sample from each subset
# ---------------------------------------------------------------------------
for subset_name, subset_root in SUBSETS.items():
    print(f'\n--- {subset_name} ---')

    if not os.path.isdir(subset_root):
        print(f'  [SKIP] Directory not found: {subset_root}')
        continue

    img_path, denoised, circles = find_sample(subset_root)

    if img_path is None:
        print('  [SKIP] No successfully segmented image found in first 100 candidates.')
        continue

    print(f'  Image   : {img_path}')
    print(f'  Circles : {circles}')

    original, denoised_img, normalized, tensor = run_pipeline(img_path, denoised, circles)
    plot_pipeline(subset_name, img_path, original, denoised_img, circles, normalized, tensor)

In [ ]:
# ---------------------------------------------------------------------------
# Tensor shape / dtype sanity check
# ---------------------------------------------------------------------------
print('Final tensor properties')
print(f'  shape : {tensor.shape}   (expected: (128, 128, 1))')
print(f'  dtype : {tensor.dtype}   (expected: float32)')
print(f'  min   : {tensor.min():.4f}  (expected >= 0.0)')
print(f'  max   : {tensor.max():.4f}  (expected <= 1.0)')

assert tensor.shape == (128, 128, 1), f'Shape mismatch: {tensor.shape}'
assert tensor.dtype == np.float32,    f'Dtype mismatch: {tensor.dtype}'
assert tensor.min() >= 0.0,           f'Values below 0: {tensor.min()}'
assert tensor.max() <= 1.0,           f'Values above 1: {tensor.max()}'
print('All assertions passed.')